# Часть 1. Проверка гипотезы в Python и составление аналитической записки

## Выполнил

- Автор: Флоря Виктория Александровна
- Дата: 02.06.26

## Цели и задачи проекта

<font color='#777778'> Цель проекта - проверка гипотезы: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении Яндекс Книги, чем пользователи из Москвы. Докажем это статистически, используя одностороннюю проверку гипотезы с двумя выборками.</font>

## Описание данных

<font color='#777778'> Таблица `bookmate.audition` содержит данные об активности пользователей и состоит из следующих полей:  
•	audition_id — уникальный идентификатор сессии чтения или прослушивания;  
•	puid — идентификатор пользователя;  
•	usage_platform_ru — название платформы, с помощью которой пользователь слушал контент;  
•	msk_business_dt_str — дата события в формате строки (московское время);  
•	app_version — версия приложения, которая использовалась для чтения или прослушивания;  
•	adult_content_flg — был ли это контент для взрослых: True или False;  
•	hours — длительность чтения или прослушивания в часах;  
•	hours_sessions_long — продолжительность длинных сессий чтения или прослушивания в часах;  
•	kids_content_flg — был ли это детский контент: True или False;  
•	main_content_id — идентификатор основного контента;  
•	usage_geo_id — идентификатор географического местоположения.  
    
Таблица `bookmate.content` содержит данные о контенте и состоит из следующих полей:  
•	main_content_id — идентификатор основного контента;  
•	main_author_id — идентификатор основного автора контента;  
•	main_content_type — тип контента;  
•	main_content_name— название контента;  
•	main_content_duration_hours — длительность контента в часах;  
•	published_topic_title_list — список жанров контента.  
    
Таблица `bookmate.author` содержит данные об авторах контента и состоит из следующих полей:  
•	main_author_id — идентификатор основного автора контента;  
•	main_author_name — имя основного автора контента.  
    
Таблица `bookmate.geo` содержит данные о местоположении и состоит из следующих полей:  
•	usage_geo_id — идентификатор географического положения;  
•	usage_geo_id_name — город или регион географического положения;  
•	usage_country_name — страна географического положения.  
</font>

## Содержимое проекта

<font color='#777778'>Часть 1. Проверка гипотезы в Python и составление аналитической записки¶  
    
1. Загрузка данных и знакомство с ними  
2. Проверка гипотезы в Python  
3. Аналитическая записка  
    
</font>

---

## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
# Загружаем библиотеку pandas
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_effectsize
from math import ceil
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
from scipy import stats

In [2]:
# Загружаем файл CSV в датафрейм df_knigi
df_knigi= pd.read_csv('https://code.s3.yandex.net/datasets/yandex_knigi_data.csv')

In [3]:
# Выводим первые 5 строк датафрейма
df_knigi.head()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


In [4]:
# Изначально строк в датафрейме
len(df_knigi)

8784

In [5]:
# Найдем дубликаты
duplicates = df_knigi['puid'].duplicated()  # Получаем булеву серию
duplicate_count = duplicates.sum()         # Считаем количество True
print(f'Дубликатов puid: {duplicate_count}')

Дубликатов puid: 244


In [6]:
# Удаляем дубликаты, оставляем уникальных пользователей (берем первую запись)
df_knigi=df_knigi.drop_duplicates(subset='puid')

In [7]:
# Делим данные по городам
moscow=df_knigi[df_knigi['city']=='Москва']['hours']
spb=df_knigi[df_knigi['city']=='Санкт-Петербург']['hours']

In [8]:
print(f'Москва: n={len(moscow)}, среднее = {moscow.mean():.2f}, std = {moscow.std():.2f}')
print(f'Санкт-Петербург: n={len(spb)}, среднее = {spb.mean():.2f}, std = {spb.std():.2f}')

Москва: n=6234, среднее = 10.88, std = 36.85
Санкт-Петербург: n=2306, среднее = 11.26, std = 39.83


*Москва:*  
кол-во пользователей - 6234    
средняя активность - 10,88  
стандартные отклонения - 36,85  

*Санкт-Петербург:*  
кол-во пользователей - 2306    
средняя активность - 11,26  
стандартные отклонения - 39,83  

Пользователей из Москвы больше - 6234, но средняя активность в Санкт-Петербурге выше - 11,26, но разница небольшая с Москвой.
Стандартные отклонения большие  (около 37 и 40), что говорит о сильном разбросе: некоторые пользователи почти не читают, а некоторые проводят сотни часов.

## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
# Односторонний тест (alternative='greater')
t_stat, p_value = stats.ttest_ind (spb,moscow, alternative = 'greater', equal_var=False)
alpha = 0.05  # Уровень значимости
print(f' t-статистика = {t_stat:.4f}')
print(f' p-value = {p_value:.6f}')
if p_value < alpha:
    print ('Отвергаем Но: среднее время в СПб статистически значимо больше, чем в Москве')
else:
    print ('Не отвергаем Но: нет достаточных доказательств, что среднее время в СПб больше')

 t-статистика = 0.4028
 p-value = 0.343571
Не отвергаем Но: нет достаточных доказательств, что среднее время в СПб больше


## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



1. Тип теста и уровень значимости  
Использован двухвыборочный независимый t-тест Уэлча (без предположения о равенстве дисперсий) с односторонней альтернативой (среднее СПб > среднего Москвы).  
·Уровень статистической значимости: α = 0.05 (стандартный порог для отвержения нулевой гипотезы).
  
2. Результат теста  
  · p-value = 0.3436 > α = 0.05  
  · t-статистика = 0.4028

3. Вывод (интерпретация)  
Нулевая гипотеза H₀ (средняя активность в СПб ≤ средней активности в Москве) не отвергается. Статистически значимых доказательств того, что пользователи из Санкт-Петербурга проводят в приложении больше часов, чем пользователи из Москвы, не получено.

4. Возможные причины полученного результата  
  · Высокая дисперсия – стандартные отклонения (~37–40 часов) намного превышают разницу средних (0.38 часа). Это типично для метрик вовлечённости: есть пассивные пользователи (0–5 часов) и несколько «суперактивных» (100+ часов).  
  · Недостаточная статистическая мощность – размер группы СПб (2306) может быть маловат, чтобы обнаружить такую маленькую разницу на фоне большого разброса. Для обнаружения эффекта в 0.38 часа потребовались бы тысячи дополнительных наблюдений.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



Цель: изучить влияние новой версии сайта интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни, на привлечение новых клиентов и увеличения числа продаж.  
Задача — провести оценку результатов A/B-теста. 

## 2. Загрузите данные, оцените их целостность.


# Описание данных  
•	https://code.s3.yandex.net/datasets/ab_test_participants.csv — таблица участников тестов.   
Структура файла:   
	`user_id` — идентификатор пользователя;  
	`group` — группа пользователя;  
	`ab_test` — название теста;  
	`device` — устройство, с которого происходила регистрация.  

•	https://code.s3.yandex.net/datasets/ab_test_events.zip — архив с одним csv-файлом, в котором собраны события 2020 года;  
Структура файла:    
	`user_id` — идентификатор пользователя;  
	`event_dt` — дата и время события;  
	`event_name` — тип события;  
	`details` — дополнительные данные о событии.  


## Содержимое проекта

<font color='#777778'>Часть 2. Анализ результатов A/B-тестирования  
1. Определим цели исследования  
2. Загрузим данные, оценим их целостность  
3. По таблице ab_test_participants оценим корректность проведения теста  
4. Проведем оценку результатов A/B-тестирования
    
</font>


In [10]:
from datetime import timedelta
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import math
import numpy as np 
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import norm

In [11]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [12]:
participants.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [13]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [14]:
events.head()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [15]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [16]:
# Найдем кол-во строк в датафреймах
print(participants.shape[0], events.shape[0])

14525 787286


Оценка целостности датафрейма `participants`

In [17]:
# Найдем дубликаты user_id в датафрейме participants
duplicates = participants['user_id'].duplicated()  # Получаем булеву серию
duplicate_count = duplicates.sum()         # Считаем количество True


# Кол-во пропусков в датафрейме participants
mission_count=participants.isnull().sum()

print(f'Дубликатов user_id: {duplicate_count}, Пропуски: {mission_count}')

Дубликатов user_id: 887, Пропуски: user_id    0
group      0
ab_test    0
device     0
dtype: int64


In [18]:
print(f"Уникальные тесты: {participants['ab_test'].unique()}")
print(f"Группы: {participants['group'].unique()}")
print(f"Устройства: {participants['device'].unique()}")

Уникальные тесты: ['interface_eu_test' 'recommender_system_test']
Группы: ['B' 'A']
Устройства: ['Mac' 'Android' 'iPhone' 'PC']


Оценка целостности датафрейма `events`

In [19]:
# Найдем дубликаты в датафрейме events
duplicates = events.duplicated()  
duplicate_count = duplicates.sum()         

# Кол-во пропусков в датафрейме events
mission_count=events.isnull().sum()
print(f'Дубликатов : {duplicate_count}, Пропуски: {mission_count}')

Дубликатов : 36318, Пропуски: user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64


In [20]:
print(f"Уникальные события: {events['event_name'].unique()}")

print(f"Диапазон дат: с {events['event_dt'].min()} по {events['event_dt'].max()}")

Уникальные события: ['End of Black Friday Ads Campaign' 'registration' 'product_page' 'login'
 'product_cart' 'purchase' 'Start of Christmas&New Year Promo'
 'Start of CIS New Year Gift Lottery']
Диапазон дат: с 2020-12-01 00:00:00 по 2020-12-31 23:59:48


In [21]:
# --- Проверка соответствия участников и событий ---
users_in_participants = set(participants['user_id'])
users_in_events = set(events['user_id'])
print(f"\n=== Согласованность ===")
print(f"Пользователей в participants: {len(users_in_participants)}")
print(f"Пользователей в events: {len(users_in_events)}")
print(f"Пересечение: {len(users_in_participants & users_in_events)}")
print(f"Пользователи participants, отсутствующие в events: {len(users_in_participants - users_in_events)}")
print(f"Пользователи events, отсутствующие в participants: {len(users_in_events - users_in_participants)}")


=== Согласованность ===
Пользователей в participants: 13638
Пользователей в events: 144184
Пересечение: 13638
Пользователи participants, отсутствующие в events: 0
Пользователи events, отсутствующие в participants: 130546


## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

1. Выделяем пользователей теста

In [22]:
# 1. Фильтруем нужный тест
test_name = 'interface_eu_test'
test_users = participants[participants['ab_test'] == test_name].copy()

print(f"Всего участников теста {test_name}: {len(test_users)}")

Всего участников теста interface_eu_test: 10850


In [23]:
# 2. Соответствие ТЗ: группы A и B
groups = test_users['group'].unique()
print(f"Группы в тесте: {sorted(groups)}")
if set(groups) == {'A', 'B'}:
    print(" Состав групп соответствует ТЗ (только A и B)")
else:
    print(" Нарушение: присутствуют другие группы")

Группы в тесте: ['A', 'B']
 Состав групп соответствует ТЗ (только A и B)


In [24]:
# 3. Дубликаты пользователей внутри теста
dups = test_users['user_id'].duplicated().sum()
print(f"Дубликатов user_id в тесте: {dups}")
if dups == 0:
    print("✓ Каждый пользователь встречается один раз")
else:
    print("✗ Нарушение: один пользователь записан несколько раз в одном тесте")

Дубликатов user_id в тесте: 0
✓ Каждый пользователь встречается один раз


In [25]:
# 4. Равномерность распределения
group_counts = test_users['group'].value_counts()
print("\nРаспределение по группам:")
display(group_counts)
group_percents = group_counts / len(test_users) * 100
print(f"В процентах:\nA: {group_percents.get('A', 0):.1f}%, B: {group_percents.get('B', 0):.1f}%")


Распределение по группам:


B    5467
A    5383
Name: group, dtype: int64

В процентах:
A: 49.6%, B: 50.4%


In [26]:
# 5. Пересечение с конкурирующим тестом
other_test = 'recommender_system_test'
other_users = set(participants[participants['ab_test'] == other_test]['user_id'])
current_users = set(test_users['user_id'])
intersection = current_users & other_users
print(f"\nПересечение с тестом '{other_test}': {len(intersection)} пользователей")
if len(intersection) > 0:
    print(f"Это составляет {len(intersection)/len(current_users)*100:.2f}% от участников теста")
    print("Обнаружено пересечение тестов – возможна интерференция эффектов")
    # Посмотрим распределение групп в пересечении
    inter_df = test_users[test_users['user_id'].isin(intersection)]
    print("Распределение групп среди пересекающихся пользователей:")
    print(inter_df['group'].value_counts())
else:
    print(" Пересечение с другими тестами отсутствует")


Пересечение с тестом 'recommender_system_test': 887 пользователей
Это составляет 8.18% от участников теста
Обнаружено пересечение тестов – возможна интерференция эффектов
Распределение групп среди пересекающихся пользователей:
B    456
A    431
Name: group, dtype: int64


3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [27]:
# 1. Фильтруем нужный тест
test_name = 'interface_eu_test'
test_users = participants[participants['ab_test'] == test_name].copy()

# 2. Получаем список user_id участников теста
user_ids = test_users['user_id'].unique().tolist()  # unique() убирает дубликаты

# 3. Оставляем только события этих пользователей
events_test = events[events['user_id'].isin(user_ids)].copy()

# 4. Исключаем системные маркетинговые события (не относятся к действиям пользователей)
marketing_events = [
    'End of Black Friday Ads Campaign',
    'Start of Christmas&New Year Promo',
    'Start of CIS New Year Gift Lottery'
]

events_test = events_test[~events_test['event_name'].isin(marketing_events)].copy()  # фильтруем события

# 5. Убираем пересечения с конкурирующим тестом
other_test = 'recommender_system_test'
other_users = set(participants[participants['ab_test'] == other_test]['user_id'])
current_users = set(test_users['user_id'])
intersection = current_users & other_users

# Удаляем из событий тех пользователей, которые есть в конкурирующем тесте
events_test = events_test[~events_test['user_id'].isin(intersection)].copy()

print(f"Событий для участников теста (без маркетинговых и пересечений): {len(events_test)}")
print(f"Уникальных пользователей: {events_test['user_id'].nunique()}")

Событий для участников теста (без маркетинговых и пересечений): 73815
Уникальных пользователей: 9963


- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [28]:
# 1. Извлекаем дату регистрации (первое событие registration на пользователя)
registration = (
    events_test[events_test['event_name'] == 'registration']
    .groupby('user_id')['event_dt']
    .min()
    .reset_index()
    .rename(columns={'event_dt': 'reg_dt'})
)

print(f"Пользователей с регистрацией: {len(registration)}")
print(f"Всего пользователей в events_test: {events_test['user_id'].nunique()}")
# Если числа не совпадают, значит у некоторых нет регистрации — их исключим

# 2. Присоединяем дату регистрации к каждому событию
events_with_reg = events_test.merge(registration, on='user_id', how='inner')

# 3. Вычисляем lifetime в днях (float)
events_with_reg['lifetime_days'] = (
    (events_with_reg['event_dt'] - events_with_reg['reg_dt']).dt.total_seconds() / (24 * 3600)
)

# 4. Оставляем события первых 7 дней (0 <= lifetime_days <= 7)
events_7days = events_with_reg[events_with_reg['lifetime_days'] <= 7].copy()

# 5. (Опционально) удаляем само событие регистрации из анализа,
#    чтобы не искажать воронку (регистрация — это старт, а не действие после неё)
events_7days = events_7days[events_7days['event_name'] != 'registration']

print(f"\n=== После фильтрации по 7 дням ===")
print(f"Количество событий: {len(events_7days)}")
print(f"Количество уникальных пользователей: {events_7days['user_id'].nunique()}")
print(f"Диапазон lifetime_days: [{events_7days['lifetime_days'].min():.2f}, {events_7days['lifetime_days'].max():.2f}]")
 

Пользователей с регистрацией: 9963
Всего пользователей в events_test: 9963

=== После фильтрации по 7 дням ===
Количество событий: 53842
Количество уникальных пользователей: 9963
Диапазон lifetime_days: [0.00, 7.00]


Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [29]:
# --- Параметры теста ---
baseline = 0.30
alpha = 0.05
power = 0.80
effect = 0.03

# Все участники теста interface_eu_test
test_users = participants[participants['ab_test'] == test_name]

# Все участники конкурирующего теста
other_users = set(participants[participants['ab_test'] == other_test]['user_id'])

# Исключаем пересекающихся пользователей
clean_users = test_users[~test_users['user_id'].isin(other_users)]

# --- Фактические размеры групп ПОСЛЕ УДАЛЕНИЯ ---
group_counts = clean_users['group'].value_counts()
nA = group_counts['A']   # 4952
nB = group_counts['B']   # 5011


z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)

p_pooled = baseline + effect / 2
se = math.sqrt(p_pooled * (1 - p_pooled) * (1/nA + 1/nB))

min_diff = (z_alpha + z_beta) * se

n_required = ((z_alpha + z_beta) ** 2 * 2 * p_pooled * (1 - p_pooled)) / (effect ** 2)
n_required = math.ceil(n_required)

print("\n=== Оценка достаточности выборки ===")
print(f"Цель: обнаружить абсолютный прирост конверсии с {baseline:.0%} до {baseline+effect:.0%}")
print(f"Фактический размер групп: A = {nA}, B = {nB}")
print(f"Необходимый размер группы (при равном распределении): {n_required}")
if nA >= n_required and nB >= n_required:
    print(" Выборка ДОСТАТОЧНА для обнаружения заданного эффекта.")
else:
    print(" Выборка НЕДОСТАТОЧНА. Нужно больше пользователей или больший эффект.")
print(f"С текущей выборкой можно обнаружить минимальный эффект = {min_diff:.2%} (абсолютного прироста).")


=== Оценка достаточности выборки ===
Цель: обнаружить абсолютный прирост конверсии с 30% до 33%
Фактический размер групп: A = 4952, B = 5011
Необходимый размер группы (при равном распределении): 3764
 Выборка ДОСТАТОЧНА для обнаружения заданного эффекта.
С текущей выборкой можно обнаружить минимальный эффект = 2.61% (абсолютного прироста).


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [30]:
# 1. Общее количество посетителей по группам (из clean_users)
total_visitors = clean_users['group'].value_counts().sort_index()
# total_visitors['A'] = 4952, total_visitors['B'] = 5011

# 2. Покупатели: пользователи, у которых есть событие 'purchase' в events_7days
buyers_users = events_7days[events_7days['event_name'] == 'purchase']['user_id'].unique()
buyers_df = pd.DataFrame({'user_id': buyers_users})

# Присоединяем группу из clean_users
buyers_with_group = buyers_df.merge(clean_users[['user_id', 'group']], on='user_id', how='left')

# Количество покупателей по группам
buyers_count = buyers_with_group['group'].value_counts().sort_index()

# 3. Формируем результат
result = pd.DataFrame({
    'group': ['A', 'B'],
    'total_visitors': [total_visitors['A'], total_visitors['B']],
    'buyers': [buyers_count.get('A', 0), buyers_count.get('B', 0)]
})
result['conversion'] = result['buyers'] / result['total_visitors']

print("=== Количество посетителей и покупателей (очищенная выборка) ===")
print(result)
print("\n=== Конверсия в покупку ===")
for _, row in result.iterrows():
    print(f"Группа {row['group']}: {row['conversion']:.2%} ({row['buyers']}/{row['total_visitors']})")

=== Количество посетителей и покупателей (очищенная выборка) ===
  group  total_visitors  buyers  conversion
0     A            4952    1377    0.278069
1     B            5011    1480    0.295350

=== Конверсия в покупку ===
Группа A: 27.81% (1377/4952)
Группа B: 29.54% (1480/5011)


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

Вывод: Всего пользователей 9963. В группе А -4952, в группе В -5011. Выборки достаточно велики.   
Распределение близко к равномерному, что позволяет сравнивать группы без значительного смещения из-за разницы в объеме. Конверсия в тестовой группе (В) выше - 29.54%, чем в контрольной (А) - 27.81%. Относительное улучшение примерно +1,73%, относительно конвесии контрольной группы.   
Прирост конверсии не соответствует запланированному приросту в 3 пп, поэтому эффект от изменений есть, но недостаточно высокий и нам предстоит определить является ли он статистически значимым. 


## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Гипотеза: Нулевая гипотеза (H₀): уровень конверсии интернет‑магазина статистически значимо не изменится после внедрения нового интерфейса  
     
Альтернативная гипотеза (H₁): уровень конверсии интернет‑магазина увеличится с внедрением нового интерфейса сайта, и это изменение будет статистически значимым
 
Уровень значимости: alpha = 0.05 

Используем z-тест для двух пропорций с односторонней альтернативой.

In [31]:
# Данные 
nA, succA = total_visitors['A'], buyers_count.get('A', 0)
nB, succB = total_visitors['B'], buyers_count.get('B', 0)
#Конверсии 
convA, convB = succA/nA, succB/nB 
print(f"Конверсия: A={convA:.2%}, B={convB:.2%}, Δ={convB-convA:.2%}\n")
# Односторонний z-тест (H1: pB > pA) 
z, p = proportions_ztest([succB, succA], [nB, nA], alternative='larger') 
print(f"z = {z:.4f}, p-value (односторонний) = {p:.6f}")
# Вывод 
alpha = 0.05 
if p < alpha: 
    print("Отвергаем H0: конверсия B статистически значимо выше A.") 
else: 
    print("Не отвергаем H0: нет доказательств, что конверсия B выше.")

Конверсия: A=27.81%, B=29.54%, Δ=1.73%

z = 1.9070, p-value (односторонний) = 0.028263
Отвергаем H0: конверсия B статистически значимо выше A.


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

Результаты: 
1. Конверсия в группе A (контрольной): 27.81 %.
2. Конверсия в группе B (тестовой): 29.54 %.
3. Разница (Δ): +1.73 % 
3. Z‑статистика: 1.9070
4. P‑value (односторонний тест):  0.028263
5. Уровень значимости (α): 0,05. - задано

# Выводы по результатам A/B-тестирования

**Цель теста:** оценить влияние изменений интерфейса на конверсию в покупку (целевое действие) пользователей, зарегистрировавшихся в декабре 2020 г.  

**Ожидаемый эффект:** абсолютный прирост конверсии на 3 процентных пункта (с 30% до 33%) – согласно исходному техническому заданию.

**Фактически получено:**  

Конверсия в группе A (контроль): 27.81%
Конверсия в группе B (тест): 29.54%
Абсолютная разница: +1.73 п.п. (относительный рост +6.2%)

**Статистическая значимость:**  

Односторонний z-тест: z = 1.907, p-value = 0.0283 (< 0.05)
Нулевая гипотеза (конверсия B ≤ конверсия A) отвергается на уровне значимости 5%.
Это означает, что наблюдаемое различие не является случайным и статистически значимо.

**Достигнут ли ожидаемый эффект?**  

Нет, ожидаемый эффект в 3 п.п. не достигнут.
Фактический прирост (1.73 п.п.) оказался меньше запланированного минимального детектируемого эффекта (MDE = 3 п.п.).

**Однако важно отметить:**   

Статистическая мощность была рассчитана на обнаружение эффекта ≥3 п.п. при заданных размерах групп (≈5000).
Мы обнаружили меньший эффект (1.73 п.п.), но при этом он всё равно оказался статистически значимым благодаря тому, что фактическая разница превысила минимальный детектируемый эффект для одностороннего теста (1.2 п.п.).
Это говорит о том, что исходный MDE = 3 п.п. был консервативным – тест «увидел» даже меньший эффект.

**Практическая значимость:**  

Хотя эффект не достиг заявленных 3 п.п., прирост в 1.73 п.п. (более 6% относительного роста) может быть бизнес-ценным в зависимости от маржинальности продукта и затрат на трафик.
Например, для крупного интернет-магазина такой рост конверсии часто считается успехом.

**Рекомендации по итогам теста:**  

Внедрить изменение интерфейса (группу B) на всю аудиторию, если бизнес готов принять эффект от 1.5–1.7 п.п. как достаточный.
Эффект статистически подтверждён, риск ложноположительного вывода низок (p<0.05).

Продолжить мониторинг после внедрения – возможно, эффект немного изменится (постоянный мониторинг необходим для подтверждения устойчивости).

Если строго требуется эффект ≥3 п.п. (например, окупаемость инвестиций в разработку не достигается при меньшем приросте), то тест не достиг цели, и изменения внедрять не стоит. В этом случае рекомендуется либо увеличить выборку, чтобы проверить, не скрывается ли за 1.73 п.п. истинный эффект 3 п.п. (но это маловероятно), либо пересмотреть гипотезу.

**Итог:**
Статистическая гипотеза: H₀ отвергнута → тестовая группа лучше контрольной.

Ожидаемый бизнес-эффект (3 п.п.) – не достигнут, но получен положительный и значимый эффект в размере +1.73 п.п., который также может быть признан успехом в зависимости от порога практической значимости, принятого в компании.

Решение о внедрении должно приниматься на основе экономической модели (ROI от дополнительных конверсий против затрат на разработку и поддержку новой версии).
